# RF-DETR Temporal и TransVOD++ в Google Colab

Notebook устанавливает оба проекта в одном GPU-runtime и измеряет чистый FPS на 300 ключевых кадрах. RF-DETR Temporal использует 3 опорных кадра, TransVOD++ Swin-B - штатные 14. Чтение изображений, предобработка, передача на GPU, постобработка рамок и фильтрация классов `car`/`bus` не входят в измеряемое время.

Перед запуском выберите **Runtime -> Change runtime type -> T4 GPU**. Выполняйте ячейки сверху вниз.

In [ ]:
import os
import platform
import subprocess
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

## 1. Google Drive и пути

Ожидаемая папка: `MyDrive/rf_detr_colab`. При необходимости измените только значения в следующей ячейке.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/rf-detr-colab')
RF_ARCHIVE = DRIVE_ROOT / 'rf-detr-temporal.zip'
TRANSVOD_ARCHIVE = DRIVE_ROOT / 'TransVOD_plusplus_swinb_car_bus_eval.zip'
DATASET_ARCHIVE = DRIVE_ROOT / 'temporal_dataset_combined.zip'
SELECTED_DATASET_ARCHIVE = DRIVE_ROOT / 'temporal_dataset_transvod_selected.zip'
RF_CHECKPOINT_SOURCE = DRIVE_ROOT / 'checkpoint_29.ckpt'
TRANSVOD_CHECKPOINT_SOURCE = DRIVE_ROOT / 'transvodpp_swinb_imagenet_vid.pth'

RF_ROOT = Path('/content/rf-detr-temporal')
TRANSVOD_ROOT = Path('/content/TransVOD_plusplus_swinb_car_bus_eval')
DATASET_ROOT = Path('/content/temporal_dataset_combined')
SELECTED_DATASET_ROOT = Path('/content/temporal_dataset_transvod_selected')

required = [RF_ARCHIVE, TRANSVOD_ARCHIVE, RF_CHECKPOINT_SOURCE, TRANSVOD_CHECKPOINT_SOURCE]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Files missing from Google Drive:\n' + '\n'.join(missing))
if not DATASET_ARCHIVE.exists() and not SELECTED_DATASET_ARCHIVE.exists():
    raise FileNotFoundError('Upload either the full or compact dataset archive.')
print('All source files found.')

In [ ]:
import shutil

def extract_archive(archive: Path, expected_root: Path) -> None:
    if expected_root.exists():
        print(f'Reusing: {expected_root}')
        return
    print(f'Extracting {archive.name} ...')
    shutil.unpack_archive(str(archive), '/content')
    if not expected_root.exists():
        raise FileNotFoundError(
            f'Archive {archive} did not create {expected_root}. '
            'Correct the corresponding *_ROOT variable in the configuration cell.'
        )

extract_archive(RF_ARCHIVE, RF_ROOT)
extract_archive(TRANSVOD_ARCHIVE, TRANSVOD_ROOT)
if DATASET_ARCHIVE.exists():
    extract_archive(DATASET_ARCHIVE, DATASET_ROOT)
else:
    print('Full dataset is absent; 300-frame benchmarks should be skipped.')
if SELECTED_DATASET_ARCHIVE.exists():
    extract_archive(SELECTED_DATASET_ARCHIVE, SELECTED_DATASET_ROOT)
else:
    print('Compact selected dataset is absent; selected inference will use the full dataset.')
    SELECTED_DATASET_ROOT = DATASET_ROOT

rf_checkpoint = RF_ROOT / 'result/colab_checkpoint/checkpoint_29.ckpt'
rf_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(RF_CHECKPOINT_SOURCE, rf_checkpoint)

transvod_checkpoint = TRANSVOD_ROOT / 'checkpoints/transvodpp_swinb_imagenet_vid.pth'
transvod_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(TRANSVOD_CHECKPOINT_SOURCE, transvod_checkpoint)

assert (SELECTED_DATASET_ROOT / 'annotations/instances_val2017.json').is_file()
assert (RF_ROOT / 'tools/benchmark_inference.py').is_file()
assert (TRANSVOD_ROOT / 'benchmark.py').is_file()
print('Projects, dataset and checkpoints are ready.')

## 2. RF-DETR Temporal

RF-DETR устанавливается в основной Python Colab. Предустановленный CUDA PyTorch сохраняется.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-e', '.[train,visual]'
], cwd=RF_ROOT / 'external/rf-detr', check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', 'tensorboard', 'opencv-python-headless'
], check=True)

In [ ]:
import sys
rf_source = str(RF_ROOT / 'external/rf-detr/src')
if rf_source not in sys.path:
    sys.path.insert(0, rf_source)

import torch
import rfdetr

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime in Colab.')
print('RF-DETR import: OK')
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
rf_output = RF_ROOT / 'reports/inference_speed/temporal_car_bus_300.json'
rf_output.parent.mkdir(parents=True, exist_ok=True)
rf_env = os.environ.copy()
rf_env.update({'DEVICE': 'cuda:0', 'PYTHON_BIN': sys.executable})
subprocess.run([
    'bash', 'commands/benchmark_temporal_car_bus_300.sh',
    str(DATASET_ROOT), str(rf_checkpoint), str(rf_output)
], cwd=RF_ROOT, env=rf_env, check=True)
print('RF-DETR result:', rf_output)

## 3. TransVOD++ Swin-B

TransVOD++ получает отдельную Conda-среду Python 3.8 с PyTorch 1.13.1 + CUDA 11.7. Установка CUDA toolkit и сборка расширения могут занять 15-30 минут.

In [ ]:
import urllib.request

conda_root = Path('/content/miniconda')
conda_exe = conda_root / 'bin/conda'
if not conda_exe.exists():
    installer = Path('/content/miniconda.sh')
    urllib.request.urlretrieve(
        'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
        installer,
    )
    subprocess.run(['bash', str(installer), '-b', '-p', str(conda_root)], check=True)
else:
    print('Reusing existing Miniconda installation.')

setup_env = os.environ.copy()
setup_env['PATH'] = f'{conda_root}/bin:' + setup_env['PATH']
setup_env['ENV_NAME'] = 'transvodpp-swinb'
subprocess.run(['bash', 'setup_ubuntu.sh'], cwd=TRANSVOD_ROOT, env=setup_env, check=True)

In [ ]:
verification = (
    "import torch; "
    "from models.ops.modules import MSDeformAttn; "
    "print('PyTorch:', torch.__version__); "
    "print('CUDA runtime:', torch.version.cuda); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0))"
)
subprocess.run([
    str(conda_exe), 'run', '--no-capture-output', '-n', 'transvodpp-swinb',
    'python', '-c', verification
], cwd=TRANSVOD_ROOT, check=True)

In [ ]:
transvod_output = TRANSVOD_ROOT / 'outputs/inference_speed/transvodpp_swinb_car_bus_300.json'
transvod_output.parent.mkdir(parents=True, exist_ok=True)
benchmark_command = (
    f'cd {TRANSVOD_ROOT} && '
    f'bash scripts/benchmark_swinb_car_bus_14refs.sh '
    f'{DATASET_ROOT} {transvod_checkpoint} {transvod_output}'
)
subprocess.run([
    str(conda_exe), 'run', '--no-capture-output', '-n', 'transvodpp-swinb',
    'bash', '-lc', benchmark_command
], check=True)
print('TransVOD++ result:', transvod_output)

## 4. Точечный тест TransVOD++ на трех кадрах

Эта ячейка выполняет только три прямых прохода: `cars26_000044`, `test_cars12_000827` и `cars18_000161`. Последнее имя автоматически сопоставляется с фактической сценой `test_cars18`. Для каждого ключевого кадра загрузчик формирует штатный контекст из 14 опорных кадров той же сцены.

In [ ]:
from IPython.display import Image as DisplayImage, display

selected_output = TRANSVOD_ROOT / 'outputs/selected_swinb_car_bus'
selected_command = (
    f'cd {TRANSVOD_ROOT} && '
    f'bash scripts/infer_selected_swinb_car_bus.sh '
    f'{SELECTED_DATASET_ROOT} {transvod_checkpoint} {selected_output} 0.30'
)
subprocess.run([
    str(conda_exe), 'run', '--no-capture-output', '-n', 'transvodpp-swinb',
    'bash', '-lc', selected_command
], check=True)

for image_name in ('cars26_000044.jpg', 'test_cars12_000827.jpg', 'cars18_000161.jpg'):
    image_path = selected_output / image_name
    print(image_name)
    display(DisplayImage(filename=str(image_path), width=1000))

## 5. Сравнение и сохранение результатов

`fps` означает число предсказаний для ключевых кадров в секунду. Не сравнивайте его с полем `input_frames_processed_per_second`: TransVOD++ читает 15 входных кадров для одного предсказания, но создает только одно предсказание ключевого кадра.

In [ ]:
import json
import pandas as pd

rf_result = json.loads(rf_output.read_text(encoding='utf-8'))
transvod_result = json.loads(transvod_output.read_text(encoding='utf-8'))
comparison = pd.DataFrame([
    {
        'model': 'RF-DETR Temporal',
        'reference_frames': rf_result['temporal_num_ref_frames'],
        'measured_images': rf_result['measured_key_images'],
        'mean_latency_ms': rf_result['latency_mean_ms'],
        'median_latency_ms': rf_result['latency_median_ms'],
        'p95_latency_ms': rf_result['latency_p95_ms'],
        'fps': rf_result['fps'],
        'peak_vram_gb': rf_result['peak_vram_gb'],
    },
    {
        'model': 'TransVOD++ Swin-B',
        'reference_frames': transvod_result['architecture']['num_ref_frames'],
        'measured_images': transvod_result['measured_key_images'],
        'mean_latency_ms': transvod_result['latency_mean_ms'],
        'median_latency_ms': transvod_result['latency_median_ms'],
        'p95_latency_ms': transvod_result['latency_p95_ms'],
        'fps': transvod_result['fps'],
        'peak_vram_gb': transvod_result['peak_vram_gb'],
    },
])
display(comparison)
comparison_path = Path('/content/inference_speed_comparison.csv')
comparison.to_csv(comparison_path, index=False)

In [ ]:
results_drive = DRIVE_ROOT / 'results'
results_drive.mkdir(parents=True, exist_ok=True)
shutil.copy2(rf_output, results_drive / rf_output.name)
shutil.copy2(transvod_output, results_drive / transvod_output.name)
shutil.copy2(comparison_path, results_drive / comparison_path.name)
shutil.copytree(selected_output, results_drive / 'selected_swinb_car_bus', dirs_exist_ok=True)
print('Results saved to:', results_drive)

## Повторный запуск

Пока runtime не удален, архивы и среды повторно распаковывать не нужно. После `Runtime -> Disconnect and delete runtime` содержимое `/content` исчезнет, поэтому установку придется выполнить снова; результаты сохраняются в Google Drive.